<a href="https://colab.research.google.com/github/Nazihbenbrahim/-SentinelAI-/blob/main/fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# 3 - Prétraitement des données + création et entraînement du modèle

# 1) Encodage des variables catégorielles en numériques (one-hot)
dataset_encoded = pd.get_dummies(
    dataset,
    columns=['sex', 'smoker', 'region'],
    drop_first=True  # évite la colinéarité parfaite
)

# 2) Split 80% / 20% en train / test
from sklearn.model_selection import train_test_split

train_dataset, test_dataset = train_test_split(
    dataset_encoded,
    test_size=0.2,
    random_state=0
)

# 3) Séparation des labels (expenses) et des features
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

# 4) Normalisation des features à partir des stats du train
train_stats = train_dataset.describe().transpose()

def normalize(df):
    return (df - train_stats['mean']) / train_stats['std']

train_dataset = normalize(train_dataset)
test_dataset = normalize(test_dataset)

# 5) Construction du modèle Keras pour la régression
def build_model():
    model = keras.Sequential([
        keras.Input(shape=(train_dataset.shape[1],)),   # couche d'entrée explicite
        layers.Dense(64, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(1)  # sortie = coût médical
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='mse',
        metrics=['mae', 'mse']
    )
    return model


# 6) Entraînement du modèle
EPOCHS = 200

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    train_labels,
    epochs=EPOCHS,
    validation_split=0.2,
    verbose=0,              # mets 1 si tu veux voir les epochs défiler
    callbacks=[early_stop]
)


In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
